In [0]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "default")
dbutils.widgets.text("volume", "medallion_demo_files")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")

source_path = f"/Volumes/{catalog}/{schema}/{volume}"
schema_location = f"/Volumes/{catalog}/{schema}/{volume}/_schemas/orders"
checkpoint_location = f"/Volumes/{catalog}/{schema}/{volume}/_checkpoints/bronze_orders"
bronze_table = f"{catalog}.{schema}.bronze_orders"

# Auto Loader incrementally discovers files. availableNow processes the current
# backlog and then stops, which is convenient for a scheduled batch-style job.
#test committt
orders_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_location)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("header", "true")
        .load(source_path)
)

(
    orders_stream.writeStream
        .option("checkpointLocation", checkpoint_location)
        .trigger(availableNow=True)
        .toTable(bronze_table)
        .awaitTermination()
)

print(f"Bronze ingestion complete: {bronze_table}")